# Task 3 — Kafka topics and event contract

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` |
| Commands | `bash scripts/init_kafka_topics.sh`; `bash scripts/capture_kafka_evidence.sh` |
| Topics | Metadata, nodes, edges, and errors. |
| Required fields | `schema_version`, `event_time`, `repo`, `commit_sha`, `run_id`, `file_id`, `file_path`. |
```

## Approach and reasoning

Commands: `bash scripts/init_kafka_topics.sh` and `bash scripts/capture_kafka_evidence.sh`. The Kafka contract separates metadata, graph nodes, graph edges, and parser errors so each downstream consumer can subscribe to only the stream it owns. Every event carries provenance fields that tie the message back to the repository, commit, run, and file.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Topic creation | The executed output records the expected Kafka topic layout. |
| Event schema | The chapter lists the common provenance fields required on each event. |
| Captured samples | The output proves node, edge, metadata, and error events were captured from the run. |


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
topics = ['cpg.nodes', 'cpg.edges', 'cpg.metadata', 'cpg.errors']
print('topics:', ', '.join(topics))
print('node events:', manifest['metrics']['kafka']['node_events'])
print('edge events:', manifest['metrics']['kafka']['edge_events'])
print('metadata/error:', manifest['metrics']['kafka']['metadata_events'], '/', manifest['metrics']['kafka']['error_events'])
print('topic_details.txt contains all four:', all(t in (ROOT / 'screenshots/kafka/topic_details.txt').read_text() for t in topics))


topics: cpg.nodes, cpg.edges, cpg.metadata, cpg.errors
node events: 21415
edge events: 7968
metadata/error: 5 / 1
topic_details.txt contains all four: True


## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | The four explicit topics were created with the expected partition layout, and node/edge IDs were unique across the captured run. |
| Failed | Provenance could previously drift between the parser and evidence capture. |
| Resolution | Capture now compares every sampled event with the exact cloned commit SHA, and the manifest hashes the raw artifacts. |
```
